### PSEUDO GENERATE ###

In [2]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

# Cek device biar inferensinya ngebut pake GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Pake device: {'GPU' if device == 0 else 'CPU'}")

# 1. Panggil Teacher Model dari Hugging Face
print("Loading Teacher Model...")
teacher_model = pipeline(
    "sentiment-analysis", 
    model="mdhugol/indonesia-bert-sentiment-classification",
    device=device
)

# 2. Load data training lu (yang 1100 data)
# Pastikan path-nya bener ya, ngikutin code lu sebelumnya
df_train = pd.read_csv("../splitting/train.csv")

# Bikin list kosong buat nampung hasil tebakan
pseudo_labels = []
pseudo_scores = []

print("Mulai proses Pseudo-Labelling...")
# 3. Looping buat nebak per baris teks
for text in tqdm(df_train["cleaned_text"].astype(str)):
    try:
        # Mesin nebak sentimen, potong max 512 karakter jaga-jaga
        result = teacher_model(text[:512]) 
        
        # Ekstrak label dan nilai ke-yakinan (confidence score)
        label_text = result[0]['label']
        score = result[0]['score']
        
        pseudo_labels.append(label_text)
        pseudo_scores.append(score)
    except Exception as e:
        print(f"Error di teks: {text} -> {e}")
        pseudo_labels.append("ERROR")
        pseudo_scores.append(0.0)

# Masukin hasil mentahan ke dataframe
df_train["teacher_label"] = pseudo_labels
df_train["confidence_score"] = pseudo_scores

# 4. Mapping Label dari Teacher ke Format Angka Lu
# Asumsi kamus lu: 0 = Negatif, 1 = Netral, 2 = Positif
def map_label(label_str):
    if label_str == "LABEL_0": 
        return 2  # mdhugol: Positif -> lu: 2
    elif label_str == "LABEL_1": 
        return 1  # mdhugol: Netral -> lu: 1
    elif label_str == "LABEL_2": 
        return 0  # mdhugol: Negatif -> lu: 0
    else: 
        return 1  # Default netral kalau ada error

# INI BAGIAN YANG UDAH DI-FIX (Pake fungsi map_label)
df_train["pseudo_label"] = df_train["teacher_label"].apply(map_label)

# Bikin dataframe baru khusus buat bahan training Model 2
df_pseudo = pd.DataFrame({
    "cleaned_text": df_train["cleaned_text"],
    "label": df_train["pseudo_label"] # Pake label hasil mapping yang bener
})

# 5. Save ke CSV baru
df_pseudo.to_csv("../data_labelling/train_pseudo.csv", index=False)
print("\nBeres, Men! Data train_pseudo.csv udah siap dipakai buat training Model 2.")

Pake device: GPU
Loading Teacher Model...


Device set to use cuda:0


Mulai proses Pseudo-Labelling...


100%|██████████| 1100/1100 [00:11<00:00, 94.68it/s]


Beres, Men! Data train_pseudo.csv udah siap dipakai buat training Model 2.


### LOAD DATA ###

In [3]:
import torch
import numpy as np
import random
import pandas as pd

from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from tqdm import tqdm

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [5]:
train_df = pd.read_csv("../data_labelling/train_pseudo.csv")
val_df   = pd.read_csv("../data_labelling/val_labeled.csv")
test_df  = pd.read_csv("../data_labelling/test_labeled.csv")

train_df["label"] = train_df["label"].astype(int)
val_df["label"]   = val_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

print("Distribusi Train:")
print(train_df["label"].value_counts())

Distribusi Train:
label
0    573
2    296
1    231
Name: count, dtype: int64


In [6]:
MODEL_NAME = "indobenchmark/indobert-large-p1"
MAX_LEN = 128
BATCH_SIZE = 8   # kecil karena large
EPOCHS = 5
LR = 2e-5
NUM_LABELS = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [8]:
class SentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["cleaned_text"].values
        self.labels = dataframe["label"].values
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return item

In [9]:
train_dataset = SentimentDataset(train_df)
val_dataset   = SentimentDataset(val_df)
test_dataset  = SentimentDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [11]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=train_df["label"].unique(),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Class Weights:", class_weights)

Class Weights: tensor([0.6399, 1.2387, 1.5873], device='cuda:0')


In [12]:
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [13]:
ACCUMULATION_STEPS = 4
best_val_loss = float('inf') # Variabel buat nyimpen loss terbaik

# Penyesuaian total_steps buat scheduler karena ada akumulasi
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

for epoch in range(EPOCHS):
    
    # ===== TRAIN =====
    model.train()
    total_train_loss = 0
    
    loop = tqdm(train_loader, leave=True)
    optimizer.zero_grad() # Pindah ke luar loop batch
    
    for step, batch in enumerate(loop):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        # Dibagi accumulation steps biar gradiennya gak meledak pas diakumulasi
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        total_train_loss += loss.item() * ACCUMULATION_STEPS
        
        # Update bobot HANYA JIKA udah mencapai accumulation steps
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad() # Reset gradien setelah update
        
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item() * ACCUMULATION_STEPS)
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # ===== VALIDATION =====
    model.eval()
    total_val_loss = 0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            loss = criterion(logits, labels)
            
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss  : {avg_val_loss:.4f}")
    print(classification_report(val_labels, val_preds))

    # ===== SAVE BEST MODEL (EARLY STOPPING LOGIC) =====
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_indobert_model.pt')
        print(f"🔥 Val Loss turun! Menyimpan model terbaik di Epoch {epoch+1}...")

print("Training Selesai!")

# ===== TESTING PAKAI MODEL TERBAIK =====
from sklearn.metrics import classification_report, confusion_matrix

print("\nLoad model terbaik untuk Testing...")
# Load bobot (weight) dari epoch yang punya Val Loss terendah
model.load_state_dict(torch.load('best_indobert_model.pt'))
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

print("=== TEST RESULT (BEST MODEL) ===")
print(classification_report(test_labels, test_preds))
print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

Epoch 1: 100%|██████████| 138/138 [14:55<00:00,  6.49s/it, loss=0.623]



Epoch 1
Train Loss: 1.0346
Val Loss  : 1.0277
              precision    recall  f1-score   support

           0       0.77      0.56      0.65       152
           1       0.22      0.50      0.31        40
           2       0.47      0.40      0.43        58

    accuracy                           0.51       250
   macro avg       0.49      0.49      0.46       250
weighted avg       0.61      0.51      0.54       250

🔥 Val Loss turun! Menyimpan model terbaik di Epoch 1...


Epoch 2: 100%|██████████| 138/138 [14:58<00:00,  6.51s/it, loss=0.272]



Epoch 2
Train Loss: 0.6086
Val Loss  : 0.9493
              precision    recall  f1-score   support

           0       0.83      0.72      0.77       152
           1       0.32      0.42      0.37        40
           2       0.52      0.59      0.55        58

    accuracy                           0.64       250
   macro avg       0.56      0.58      0.56       250
weighted avg       0.68      0.64      0.66       250

🔥 Val Loss turun! Menyimpan model terbaik di Epoch 2...


Epoch 3: 100%|██████████| 138/138 [14:53<00:00,  6.48s/it, loss=0.0914]



Epoch 3
Train Loss: 0.2699
Val Loss  : 1.0804
              precision    recall  f1-score   support

           0       0.84      0.78      0.81       152
           1       0.31      0.38      0.34        40
           2       0.56      0.57      0.56        58

    accuracy                           0.67       250
   macro avg       0.57      0.58      0.57       250
weighted avg       0.69      0.67      0.68       250



Epoch 4: 100%|██████████| 138/138 [14:47<00:00,  6.43s/it, loss=0.0171]



Epoch 4
Train Loss: 0.1112
Val Loss  : 1.2890
              precision    recall  f1-score   support

           0       0.87      0.73      0.79       152
           1       0.30      0.38      0.33        40
           2       0.51      0.64      0.57        58

    accuracy                           0.65       250
   macro avg       0.56      0.58      0.57       250
weighted avg       0.69      0.65      0.67       250



Epoch 5: 100%|██████████| 138/138 [14:46<00:00,  6.43s/it, loss=0.0137] 



Epoch 5
Train Loss: 0.0520
Val Loss  : 1.3978
              precision    recall  f1-score   support

           0       0.85      0.71      0.77       152
           1       0.27      0.40      0.32        40
           2       0.53      0.59      0.56        58

    accuracy                           0.63       250
   macro avg       0.55      0.57      0.55       250
weighted avg       0.68      0.63      0.65       250

Training Selesai!

Load model terbaik untuk Testing...
=== TEST RESULT (BEST MODEL) ===
              precision    recall  f1-score   support

           0       0.76      0.86      0.81       130
           1       0.41      0.32      0.36        60
           2       0.63      0.60      0.62        60

    accuracy                           0.67       250
   macro avg       0.60      0.59      0.59       250
weighted avg       0.65      0.67      0.65       250

Confusion Matrix:
[[112  11   7]
 [ 27  19  14]
 [  8  16  36]]


In [14]:
from sklearn.metrics import classification_report, confusion_matrix
model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

print("=== TEST RESULT ===")
print(classification_report(test_labels, test_preds))
print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

=== TEST RESULT ===
              precision    recall  f1-score   support

           0       0.76      0.86      0.81       130
           1       0.41      0.32      0.36        60
           2       0.63      0.60      0.62        60

    accuracy                           0.67       250
   macro avg       0.60      0.59      0.59       250
weighted avg       0.65      0.67      0.65       250

Confusion Matrix:
[[112  11   7]
 [ 27  19  14]
 [  8  16  36]]


In [15]:
model.save_pretrained("./model2_pseudo")
tokenizer.save_pretrained("./model2_pseudo")

('./model2_pseudo\\tokenizer_config.json',
 './model2_pseudo\\special_tokens_map.json',
 './model2_pseudo\\vocab.txt',
 './model2_pseudo\\added_tokens.json',
 './model2_pseudo\\tokenizer.json')

### CIHUY ###

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_path = "./model2_pseudo"

tokenizer = AutoTokenizer.from_pretrained(load_path)
model = AutoModelForSequenceClassification.from_pretrained(load_path)

model.to(device)
model.eval()

print("Model loaded successfully!")

Model loaded successfully!
